# 🤖 Machine Learning Predictions Notebook

---

## Objective
Build machine learning models to predict food price inflation, enabling proactive policy responses and resource allocation.

This notebook evaluates **Hypothesis 5 (H5)**:
> *Can machine learning models predict food price inflation with reasonable accuracy?*

We test this by training three models on historical data and evaluating their generalisation on a held-out test set using R², MAE, and RMSE.

## Why Machine Learning?
Statistical analysis tells us **what happened** and **why patterns exist**. Machine learning goes further to predict **what will happen next**.

### Business Value
- **Government agencies**: Plan food assistance programs
- **Businesses**: Optimise supply chain and pricing strategies
- **Consumers**: Anticipate price changes for budgeting
- **NGOs**: Allocate resources to regions at risk

## Models We'll Build

| Model | Type | Why Use It? |
|-------|------|-------------|
| Linear Regression | Baseline | Simple, interpretable, establishes benchmark |
| Random Forest | Ensemble | Captures non-linear relationships, handles feature interactions |
| XGBoost | Gradient Boosting | State-of-the-art performance, handles missing values |

## Prerequisites
Run `Data_Cleaning.ipynb` and `Hypothesis_Testing.ipynb` first.


---
## Step 1: Import Libraries

### Libraries Used:
- **scikit-learn**: Industry-standard ML library for model building and evaluation
- **xgboost**: High-performance gradient boosting library
- **joblib**: Model serialisation for deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# XGBoost
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries loaded successfully!")

---
## Step 2: Load and Prepare Data

### Feature Engineering for ML
We need to create features that the model can learn from:
- **Lag features**: Previous month's values (time series prediction)
- **Rolling statistics**: Moving averages to capture trends
- **Encoded categories**: Transform country names to numbers

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

# Create directories for outputs
os.makedirs('../outputs/models', exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

print("✅ Data loaded!")
print(f"📊 {df.shape[0]:,} records, {df.shape[1]} columns")
print(f"🌍 {df['country'].nunique()} countries")
df.head()

In [ ]:
# Sort by country and date for proper lag calculation
df = df.sort_values(['country', 'date']).reset_index(drop=True)

# Create lag features (previous month's values)
for lag in [1, 3, 6, 12]:
    df[f'close_lag_{lag}'] = df.groupby('country')['close'].shift(lag)
    df[f'inflation_lag_{lag}'] = df.groupby('country')['inflation'].shift(lag)

# Create rolling mean features (3-month and 6-month averages)
df['close_rolling_3'] = df.groupby('country')['close'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['close_rolling_6'] = df.groupby('country')['close'].transform(lambda x: x.rolling(6, min_periods=1).mean())
df['close_rolling_12'] = df.groupby('country')['close'].transform(lambda x: x.rolling(12, min_periods=1).mean())

# Year-over-Year change in close price
df['close_yoy_change'] = df.groupby('country')['close'].pct_change(12) * 100

# Encode country as numeric
le = LabelEncoder()
df['country_encoded'] = le.fit_transform(df['country'])

# Save encoder for later use
joblib.dump(le, '../outputs/models/country_encoder.joblib')

print("✅ Feature engineering complete!")
print(f"📊 New dataset shape: {df.shape}")

---
## Step 3: Prepare Training Data

### Target Variable
We'll predict **inflation** (year-over-year food price change percentage).

### Train-Test Split Strategy
**Time-based split** (not random): We train on earlier data and test on recent data to simulate real-world prediction scenarios.

### Why Not Random Split?
For time series data, random splitting would allow the model to "cheat" by learning from future data. A temporal split ensures we're testing on truly unseen future data.

In [ ]:
# Drop rows with missing values (due to lag features)
df_ml = df.dropna(subset=['inflation', 'close_lag_12', 'inflation_lag_12']).copy()

print(f"📊 Records for ML: {len(df_ml):,} (dropped {len(df) - len(df_ml):,} with missing lags)")

# Define features
feature_cols = [
    'open', 'high', 'low', 'close',
    'price_range', 'price_change', 'price_change_pct',
    'year', 'month', 'quarter',
    'country_encoded',
    'close_lag_1', 'close_lag_3', 'close_lag_6', 'close_lag_12',
    'inflation_lag_1', 'inflation_lag_3', 'inflation_lag_6', 'inflation_lag_12',
    'close_rolling_3', 'close_rolling_6', 'close_rolling_12'
]

X = df_ml[feature_cols]
y = df_ml['inflation']

# Time-based split: use last 20% of data for testing
split_idx = int(len(df_ml) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"\n📈 Training set: {len(X_train):,} records")
print(f"📉 Test set: {len(X_test):,} records")

# Store test dates for later visualisation
test_dates = df_ml.iloc[split_idx:]['date']
test_countries = df_ml.iloc[split_idx:]['country']

In [ ]:
# Scale features for models that benefit from normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for deployment
joblib.dump(scaler, '../outputs/models/feature_scaler.joblib')

print("✅ Data scaled and split!")

---
## Step 4: Build and Evaluate Models

### Evaluation Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **R² Score** | 1 - (SS_res/SS_tot) | % of variance explained (higher is better, max=1) |
| **MAE** | Mean(\|actual - predicted\|) | Average absolute error in same units as target |
| **RMSE** | √Mean((actual - predicted)²) | Penalises large errors more than MAE |

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Train model and return evaluation metrics."""
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    metrics = {
        'Model': name,
        'Train R²': r2_score(y_train, y_pred_train),
        'Test R²': r2_score(y_test, y_pred_test),
        'Test MAE': mean_absolute_error(y_test, y_pred_test),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test))
    }
    
    return metrics, y_pred_test

# Store results
results = []
predictions = {}

### Model 1: Linear Regression (Baseline)

**Why Linear Regression?**
- Simple and interpretable
- Establishes a baseline to beat
- Fast to train
- Coefficients show feature importance

In [ ]:
print("="*60)
print("🔵 Model 1: Linear Regression (Baseline)")
print("="*60)

lr_model = LinearRegression()
lr_metrics, lr_pred = evaluate_model('Linear Regression', lr_model, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(lr_metrics)
predictions['Linear Regression'] = lr_pred

print(f"\n📊 Results:")
print(f"   Train R²: {lr_metrics['Train R²']:.4f}")
print(f"   Test R²:  {lr_metrics['Test R²']:.4f}")
print(f"   Test MAE: {lr_metrics['Test MAE']:.4f}%")
print(f"   Test RMSE: {lr_metrics['Test RMSE']:.4f}%")

### Model 2: Random Forest

**Why Random Forest?**
- Ensemble of decision trees (reduces overfitting)
- Captures non-linear relationships
- Handles feature interactions automatically
- Provides feature importance rankings

In [ ]:
print("="*60)
print("🌲 Model 2: Random Forest")
print("="*60)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_metrics, rf_pred = evaluate_model('Random Forest', rf_model, X_train, X_test, y_train, y_test)
results.append(rf_metrics)
predictions['Random Forest'] = rf_pred

print(f"\n📊 Results:")
print(f"   Train R²: {rf_metrics['Train R²']:.4f}")
print(f"   Test R²:  {rf_metrics['Test R²']:.4f}")
print(f"   Test MAE: {rf_metrics['Test MAE']:.4f}%")
print(f"   Test RMSE: {rf_metrics['Test RMSE']:.4f}%")

### Model 3: XGBoost

**Why XGBoost?**
- State-of-the-art gradient boosting algorithm
- Excellent performance on tabular data
- Built-in regularisation prevents overfitting
- Handles missing values natively
- Often wins ML competitions

In [ ]:
print("="*60)
print("🚀 Model 3: XGBoost")
print("="*60)

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_metrics, xgb_pred = evaluate_model('XGBoost', xgb_model, X_train, X_test, y_train, y_test)
results.append(xgb_metrics)
predictions['XGBoost'] = xgb_pred

print(f"\n📊 Results:")
print(f"   Train R²: {xgb_metrics['Train R²']:.4f}")
print(f"   Test R²:  {xgb_metrics['Test R²']:.4f}")
print(f"   Test MAE: {xgb_metrics['Test MAE']:.4f}%")
print(f"   Test RMSE: {xgb_metrics['Test RMSE']:.4f}%")

---
## Step 5: Compare Models

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame(results)

print("="*70)
print("📋 MODEL COMPARISON SUMMARY")
print("="*70)
print(results_df.to_string(index=False))

# Find best model
best_idx = results_df['Test R²'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"\n🏆 Best Model: {best_model_name} (R² = {results_df.loc[best_idx, 'Test R²']:.4f})")

# Save results
results_df.to_csv('../outputs/reports/ml_model_comparison.csv', index=False)
print("\n💾 Saved: outputs/reports/ml_model_comparison.csv")

In [ ]:
# Visualise model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparison
models = results_df['Model']
x = np.arange(len(models))
width = 0.35

axes[0].bar(x - width/2, results_df['Train R²'], width, label='Train', color='steelblue')
axes[0].bar(x + width/2, results_df['Test R²'], width, label='Test', color='coral')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Score by Model', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15)
axes[0].legend()
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Error comparison
axes[1].bar(x - width/2, results_df['Test MAE'], width, label='MAE', color='mediumpurple')
axes[1].bar(x + width/2, results_df['Test RMSE'], width, label='RMSE', color='seagreen')
axes[1].set_ylabel('Error (%)')
axes[1].set_title('Prediction Error by Model', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15)
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/ml_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/ml_model_comparison.png")

---
## Step 6: Feature Importance Analysis

Understanding which features drive predictions helps:
- Validate that the model is learning sensible patterns
- Identify key factors influencing inflation
- Inform policy decisions

In [ ]:
# Get feature importance from best tree-based model
importance_rf = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(importance_rf)))
ax.barh(importance_rf['Feature'], importance_rf['Importance'], color=colors)
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance (Random Forest)', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Top 5 Most Important Features:")
for i, row in importance_rf.tail(5).iterrows():
    print(f"   • {row['Feature']}: {row['Importance']:.4f}")

---
## Step 7: Predictions Visualisation

In [ ]:
# Actual vs Predicted scatter plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, pred, alpha=0.3, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
    ax.set_xlabel('Actual Inflation (%)')
    ax.set_ylabel('Predicted Inflation (%)')
    r2 = r2_score(y_test, pred)
    ax.set_title(f'{name}\nR² = {r2:.4f}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/actual_vs_predicted.png")

---
## Step 8: Save Best Model for Deployment

We'll save the best performing model for use in the Streamlit dashboard.

In [ ]:
# Save the best model (XGBoost or Random Forest based on performance)
best_model = xgb_model if 'XGBoost' in best_model_name else rf_model

# Re-train on all data for deployment
best_model.fit(X, y)

# Save model
joblib.dump(best_model, '../outputs/models/best_model.joblib')
joblib.dump(feature_cols, '../outputs/models/feature_columns.joblib')

print("="*60)
print("💾 MODEL SAVED FOR DEPLOYMENT")
print("="*60)
print(f"Model: {best_model_name}")
print(f"Location: outputs/models/best_model.joblib")
print(f"Features: outputs/models/feature_columns.joblib")
print(f"Scaler: outputs/models/feature_scaler.joblib")
print(f"Country Encoder: outputs/models/country_encoder.joblib")

---
## Summary and Conclusions

### Model Performance Summary

| Aspect | Finding |
|--------|--------|
| **Best Model** | Determined by highest Test R² score |
| **Key Features** | Previous inflation values (lag features) are most predictive |
| **Prediction Error** | MAE indicates average error in percentage points |

### Key Insights

1. **Lag features dominate**: Past inflation is the best predictor of future inflation (autoregressive nature)
2. **Tree-based models outperform**: Random Forest and XGBoost capture non-linear patterns better than Linear Regression
3. **Country matters**: The encoded country variable shows importance, confirming regional variation

### Model Limitations

- **External shocks**: Cannot predict unexpected events (pandemics, wars, natural disasters)
- **Structural changes**: Models assume patterns continue; regime changes may invalidate predictions
- **Data quality**: Predictions are only as good as the input data

### Deployment
The trained model is saved and ready for use in the Streamlit dashboard (`app.py`).

### Files Created
- `outputs/models/best_model.joblib` - Trained prediction model
- `outputs/models/feature_scaler.joblib` - Feature normalisation
- `outputs/models/country_encoder.joblib` - Country name → number mapping
- `outputs/models/feature_columns.joblib` - List of features in correct order
- `outputs/figures/ml_model_comparison.png` - Model comparison chart
- `outputs/figures/feature_importance.png` - Feature importance ranking
- `outputs/figures/actual_vs_predicted.png` - Prediction accuracy plots
- `outputs/reports/ml_model_comparison.csv` - Model metrics table